> **Interactive lesson:** Run the code cells, change inputs, and record your observations in the learner cells. This notebook is generated from the [Markdown source](11-self-supervised-learning.md); edit that source and rebuild rather than editing generated cells by hand.


# 0.11 — Self-Supervised Learning

**Depth: LEARN / REFRESH**

**Goal:** distinguish supervision sources, derive the main self-supervised objective families, and explain how BERT, GPT, and DINO learn without manual class labels.

[Month 0 roadmap](../README.md) · [Previous: Representation Learning](10-representation-learning.ipynb) · [Next: NLP](12-basic-nlp-concepts.ipynb)


## Where the targets come from

Supervision describes the learning signal, not whether a human ever touched the data.

| Setting | Target source | Example |
|---|---|---|
| Supervised | External labels associated with inputs | Human-labeled image classes |
| Unsupervised | No explicit target labels; model data structure | Clustering |
| Self-supervised | Targets or relations derived from the data itself | Predict hidden tokens |

Self-supervised learning is often treated as a subset of unsupervised learning, but the distinction is useful: it creates a predictive task from unlabeled data. Data curation can still involve human decisions, filtering, or metadata. “Self-supervised” does not mean free of bias or provenance requirements.


## Predictive and reconstruction objectives

A pretext task hides, corrupts, or separates part of an example and asks the model to predict a related target. Useful learning occurs when success requires reusable structure rather than a trivial shortcut.


### Autoregressive prediction

GPT-style language modeling factors a sequence from left to right:

```text
loss = −Σt log P(xt | x<t)
```

Every next token supplies a target. The causal objective naturally supports generation because inference uses the same direction of conditioning, though generated prefixes come from the model rather than the dataset.


### Masked prediction

BERT-style masked language modeling selects input tokens, corrupts their visible form, and predicts their original identities using context from both sides. Loss applies only to selected prediction positions. The model gains bidirectional representations but does not directly learn ordinary left-to-right generation.

Masking every selected token with the same marker creates a mismatch because that marker is absent in many downstream inputs. BERT's original corruption recipe sometimes retains or replaces selected tokens; the broader lesson is to inspect the exact data transformation rather than reduce masked learning to one slogan.

Masked autoencoders apply related logic to image patches: encode the visible subset and reconstruct missing content. Reconstruction can overemphasize low-level detail unless architecture and target representation guide the model toward useful semantics.


## Contrastive learning

> **Read next:** [A Simple Framework for Contrastive Learning of Visual Representations (SimCLR)](https://arxiv.org/abs/2002.05709) is a clean reference for augmented positive pairs, in-batch negatives, and the contrastive loss.

Contrastive learning defines positive pairs that should be similar and negatives that should be distinguishable. In a batch with paired representations `zi` and `zj`, an InfoNCE-style term is:

```text
−log [ exp(sim(zi,zj)/temperature)
       / Σk exp(sim(zi,zk)/temperature) ]
```

The denominator's candidate set defines the classification problem. Temperature controls distribution sharpness. Larger negative sets can improve discrimination but increase compute and the chance of false negatives—different items that are semantically valid matches.

Augmentation defines invariance. Treating two crops as positives teaches the representation to ignore some crop-specific details. If those details determine the downstream label, the augmentation damages utility. SimCLR uses augmented views and negatives; CLIP uses paired images and text, with other batch pairs acting as negatives.


## Teacher-student learning and DINO

DINO creates global and local views of an image. A student learns to match a teacher's probability distribution across compatible views. The teacher is updated using an exponential moving average of student parameters rather than ordinary gradient descent through the teacher.

```text
image → multiple crops ──→ student → distribution ──→ matching loss
                    └────→ teacher → target distribution
                                ↑
                        moving-average update
```

Without countermeasures, both networks could emit the same constant representation for every image: collapse. DINO combines centering and different teacher/student temperature behavior to stabilize useful targets. Other non-contrastive methods use predictors, stop-gradient operations, variance/covariance terms, or architectural asymmetry. Absence of explicit negatives does not remove the need to prevent collapse.


## Context learning versus parameter learning

Self-supervised pretraining updates weights using gradient descent. In-context learning later changes model behavior through tokens supplied at inference without updating weights. Both can appear to “learn from examples,” but their persistence and mechanisms differ.

Fine-tuning modifies parameters for a downstream objective. A linear probe freezes the encoder and measures accessible information. Prompting conditions the existing model for one context. Keep these adaptation modes separate when reporting a result.


## Evaluation and failure modes

Evaluate both the pretext objective and transferred utility. A lower reconstruction loss can preserve details irrelevant to classification. Contrastive loss can improve while nearest-neighbor semantics remain biased. Useful checks include linear probing, k-nearest-neighbor classification, retrieval, full fine-tuning, and robustness slices.

Prevent duplicates across splits and audit augmentations. Dataset scale can amplify copyright, privacy, representation, and contamination problems. Track model/data versions so downstream findings remain attributable.


## Checkpoint

1. What makes an objective self-supervised?
2. Why does masked language modeling support bidirectional context but not directly teach causal generation?
3. What do positives, negatives, and temperature do in contrastive learning?
4. Why can a teacher-student model collapse, and how does DINO reduce that risk?


<details>
<summary>Show answers</summary>

1. The predictive targets or relationships are constructed from the data rather than supplied as external task labels.
2. A masked token is predicted using visible tokens on both sides. Ordinary generation cannot condition on future tokens that have not been produced.
3. Positives define desired invariance/alignment, negatives define distinctions, and temperature changes the sharpness and gradients of the similarity distribution.
4. Matching alone admits a constant-output solution. DINO uses a moving-average teacher, centering, and temperature choices to keep targets informative.

</details>


## Exercise — Compare three objectives

For a corpus of product images and descriptions, design one autoregressive, one masked-prediction, and one contrastive pretraining task. State the representation each is likely to help, one shortcut, and a downstream evaluation.


In [ ]:
# Your work here


<details>
<summary>Show exercise solution</summary>

- Autoregressive: predict description tokens from earlier tokens, optionally conditioned on image features. It supports generation; boilerplate text may be an easy shortcut. Evaluate held-out description quality and factual grounding.
- Masked prediction: reconstruct selected text tokens or image patches from visible context. It supports bidirectional features; local texture or neighboring-token leakage may dominate. Evaluate frozen probes and retrieval on attributes requiring global context.
- Contrastive: align paired product images and descriptions while separating other pairs. It supports cross-modal retrieval; batch items describing the same product type can become false negatives. Evaluate recall@k on deduplicated products and hard category-matched negatives.

Use product-level splits so alternate photos or duplicate descriptions do not cross train and evaluation.

</details>


## Completion criteria

Classify supervision correctly, explain autoregressive/masked/contrastive/teacher-student objectives, identify collapse and shortcut risks, and choose transfer evaluations.


## Primary references

- [BERT](https://arxiv.org/abs/1810.04805)
- [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)
- [A Simple Framework for Contrastive Learning of Visual Representations](https://arxiv.org/abs/2002.05709)
- [Emerging Properties in Self-Supervised Vision Transformers](https://arxiv.org/abs/2104.14294)
- [Masked Autoencoders Are Scalable Vision Learners](https://arxiv.org/abs/2111.06377)
